# 08 · Student Candidate — MobileNetV3-Small (from scratch)

MobileNetV3-Small architecture trained from random (Kaiming) initialization.
~2.5M parameters. Same architecture as notebook 09, but no pretrained weights.

## Role in the 2×2 comparison

| | Scratch | Pretrained |
|---|---|---|
| **MobileNetV2** | notebook 06 | notebook 07 |
| **MobileNetV3** | **← this notebook** | notebook 09 |

**Two claims this notebook helps answer:**

1. *Architecture effect (scratch):* Does V3 beat V2 when both train from scratch?
   Compare this notebook against notebook 06 (V2 scratch).

2. *Initialization effect on V3:* Does pretraining improve MobileNetV3 on VWW?
   Compare this notebook against notebook 09 (same architecture, pretrained weights).

**Expected outcome:** This model underperforms notebook 09 (V3 pretrained),
confirming that ImageNet initialization provides meaningful benefit even for
a binary classification task on 7,000 images.


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import (
    MobileNetV2_Scratch, MobileNetV2_Pretrained,
    MobileNetV3_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, STUDENT_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_multi_seed, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


## Standardized hyperparameters

All four student models use **identical training conditions** to ensure the
comparison is controlled — only architecture and initialization vary.

| Parameter | Scratch models | Pretrained models |
|-----------|---------------|-------------------|
| Batch size | 64 | 64 |
| Optimizer | Adam | Adam |
| Weight decay | 1e-4 | 1e-4 |
| Label smoothing | 0.1 | 0.1 |
| Augmentation | standard | standard |
| Scheduler | CosineAnnealingLR | CosineAnnealingLR |
| Patience | 10 | 10 |
| Seeds | [41, 52, 63] | [41, 52, 63] |
| Max epochs | 50 | 25 |
| LR | 1e-3 | 3e-4 (head) → 1e-4 (full) |


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")


In [ ]:
results, best = train_multi_seed(
    model_fn     = MobileNetV3_Scratch,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63],
    save_dir     = SAVE_DIR,
    name_prefix  = "mobilenetv3_scratch",
    pretrained   = False,
    # Standard scratch hyperparameters
    epochs          = 50,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)


In [ ]:
plot_history(best, title=f"MobileNetV3-Small Scratch (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nMobileNetV3-Small Scratch")
print(f"  Mean ± Std : {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")
print(f"  Best       : {best['best_acc']*100:.2f}% @ epoch {best['best_epoch']} (seed {best['seed']})")
print(f"  Checkpoint : {best['save_path']}")


In [ ]:
# ── Quick parameter/size summary ────────────────────────────────────
m = MobileNetV3_Scratch()
total, _ = count_params(m)
size = model_size_mb(m)
print(f"MobileNetV3-Small Scratch  |  Params: {total/1e6:.2f}M  |  Size: {size:.2f}MB")
print(f"Same parameter count as V3 Pretrained (nb 09) — only initialization differs.")
